# Text to speech

[Back to the introduction](ch01_demo.ipynb). Run in order. `LLM.pipeline("text-to-speech", model=...)` turns text into speech in the browser. The model here is `Xenova/mms-tts-eng` (Meta's MMS English voice, about 140 MB quantized), which is released under **CC-BY-NC-4.0: non-commercial use only**. The result is a `Speech` object: evaluating it in a cell shows an audio player, `samples` holds the waveform and `to_wav()` returns a WAV file.

In [ ]:
from lmlite import LLM

tts = await LLM.pipeline("text-to-speech", model="mms-tts-eng", backend="wasm")
speech = await tts("Hello from the browser. This voice is generated on your own machine.")
print(speech)
speech

## The samples

`samples` is a NumPy `float32` array when NumPy is installed, else an `array.array("f")`; both are mono at `sampling_rate` Hz, with values around [-1, 1]. `to_wav()` encodes 16-bit PCM, ready to save or upload.

In [ ]:
print(type(speech.samples).__name__, len(speech.samples), "samples at", speech.sampling_rate, "Hz")
print(f"{speech.duration:.2f} s, peak {max(abs(float(x)) for x in speech.samples):.2f}")
wav = speech.to_wav()
print(len(wav), "bytes of WAV:", wav[:4], wav[8:12])

## Many texts

A text-to-speech model takes one text per call, so `map()` runs them one after another in JavaScript and reports progress after each. It returns one `Speech` per text.

In [ ]:
from IPython.display import display

sentences = ["The first sentence.", "A second, slightly longer sentence.", "And a third one."]
progress = []
clips = await tts.map(sentences, progress_callback=lambda done, total: progress.append(f"{done}/{total}"))
print("progress:", progress)
for sentence, clip in zip(sentences, clips):
    print(f"{clip.duration:.2f} s: {sentence}")
    display(clip)

Release the model; cached files are retained for subsequent runs.

In [ ]:
await tts.dispose()